In [1]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import numpy as np
import urllib.request
import os
import time
import threading
import platform
import subprocess
from collections import deque
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image

try:
    import winsound
except:
    winsound = None

print("Imports geladen!")

ALARM_PATH = "/Users/nachatissa/Desktop/SCHOOL/SEM2/Advanced AI/untitled folder/Advanced-AI/Model/mixkit-alert-alarm-1005.wav"
PAUSE_ALARM_PATH = "/Users/nachatissa/Desktop/SCHOOL/SEM2/Advanced AI/untitled folder/Advanced-AI/Model/mixkit-classic-alarm-995.wav"

if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("🚀 MPS!")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("🚀 CUDA!")
else:
    device = torch.device("cpu")
    print("CPU")

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64 * 28 * 28, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 2)
        )
    def forward(self, x):
        return self.model(x)

eyes_model = SimpleCNN().to(device)
eyes_model.load_state_dict(torch.load(
    "/Users/nachatissa/Desktop/SCHOOL/SEM2/Advanced AI/untitled folder/Advanced-AI/Model/eyes/fatigue_model.pth",
    map_location=device
))
eyes_model.eval()
print("✅ Eyes OK")

yawn_model = models.efficientnet_b0(weights="DEFAULT")
yawn_model.classifier[1] = nn.Linear(yawn_model.classifier[1].in_features, 2)
yawn_model.load_state_dict(torch.load(
    "/Users/nachatissa/Desktop/SCHOOL/SEM2/Advanced AI/untitled folder/Advanced-AI/Model/yawn/yawn_model.pth",
    map_location=device
))
yawn_model.to(device)
yawn_model.eval()
print("✅ Yawn OK")

eyes_transform = transforms.Compose([transforms.ToTensor()])
yawn_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

CONF_THRESHOLD = 0.60
YAWN_THRESHOLD = 0.62
CLOSED_ALARM_SECONDS = 1.2
INFERENCE_SKIP_FRAMES = 2
BLINK_STABLE_FRAMES = 1
YAWN_STABLE_FRAMES = 2

BLINK_WINDOW_SECONDS = 60
YAWN_WINDOW_SECONDS = 60
BLINK_LOW_THRESHOLD = 6
YAWN_HIGH_THRESHOLD = 2

YAW_THRESHOLD = 28
PITCH_THRESHOLD = 15
SMOOTH_N = 5
WARMUP_SECONDS = 3.0

blink_count = 0
yawn_count = 0
eyes_closed_since = None
was_closed_prev = False
was_yawn_prev = False

cached_eyes_closed = False
cached_is_yawn = False

stable_eye_state = None
stable_eye_frames = 0
stable_yawn_state = None
stable_yawn_frames = 0

fps = 0.0
fps_n = 0
fps_t0 = time.time()
app_start_time = time.time()

alarm_active = False
alarm_start_time = None
sleep_alarm_played = False
distraction_since = None

pause_msg_until = 0.0
distraction_msg_until = 0.0
pause_alert_active = False
pause_alert_played = False
pause_alert_until = 0.0
pause_sound_cooldown_until = 0.0

blink_times = deque()
yawn_times = deque()
yaw_history = deque(maxlen=SMOOTH_N)
pitch_history = deque(maxlen=SMOOTH_N)

last_face_box = None
face_hold_frames = 0

MODEL_PATH = "face_landmarker.task"
MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/"
    "face_landmarker/face_landmarker/float16/1/face_landmarker.task"
)

if not os.path.exists(MODEL_PATH):
    print("Downloading face landmarker model (~30 MB)...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print("Download complete.")

BaseOptions = python.BaseOptions
FaceLandmarker = vision.FaceLandmarker
FaceLandmarkerOptions = vision.FaceLandmarkerOptions
VisionRunningMode = vision.RunningMode

options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=VisionRunningMode.VIDEO,
    num_faces=1,
    min_face_detection_confidence=0.5,
    min_face_presence_confidence=0.5,
    min_tracking_confidence=0.5
)
detector = FaceLandmarker.create_from_options(options)

LANDMARK_INDICES = [1, 152, 33, 263, 61, 291]
MODEL_POINTS = np.array([
    (0.0, 0.0, 0.0),
    (0.0, -330.0, -65.0),
    (-225.0, 170.0, -135.0),
    (225.0, 170.0, -135.0),
    (-150.0, -150.0, -125.0),
    (150.0, -150.0, -125.0),
], dtype=np.float64)

def play_alarm_file(path, seconds=1.0):
    try:
        if platform.system() == "Darwin":
            p = subprocess.Popen(["afplay", path], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            time.sleep(seconds)
            p.terminate()
            try:
                p.wait(timeout=0.2)
            except:
                p.kill()
        else:
            if winsound:
                winsound.PlaySound(path, winsound.SND_FILENAME | winsound.SND_ASYNC)
                time.sleep(seconds)
                winsound.PlaySound(None, winsound.SND_PURGE)
            else:
                print("\a", end="", flush=True)
                time.sleep(seconds)
    except:
        print("\a", end="", flush=True)
        time.sleep(seconds)

def start_alarm(path, seconds=1.0):
    threading.Thread(target=play_alarm_file, args=(path, seconds), daemon=True).start()

def clamp(v, lo, hi):
    return max(lo, min(v, hi))

def head_pose_from_landmarks(lm, w, h):
    image_points = np.array([(lm[i].x * w, lm[i].y * h) for i in LANDMARK_INDICES], dtype=np.float64)
    focal = w
    camera_matrix = np.array([[focal, 0, w / 2],
                              [0, focal, h / 2],
                              [0, 0, 1]], dtype=np.float64)
    _, rvec, _ = cv2.solvePnP(MODEL_POINTS, image_points, camera_matrix, np.zeros((4, 1)))
    rot_mat, _ = cv2.Rodrigues(rvec)
    pitch, yaw, roll = cv2.RQDecomp3x3(rot_mat)[0]
    if pitch < -90:
        pitch = 180 + pitch
    elif pitch > 90:
        pitch = 180 - pitch
    return float(pitch), float(yaw), float(roll)

cap = cv2.VideoCapture(0, cv2.CAP_AVFOUNDATION)
if not cap.isOpened():
    print("Cannot open camera")
    raise SystemExit

cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1000)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 750)
cap.set(cv2.CAP_PROP_FPS, 30)

print("🎥 Drowsiness + Distraction Monitor | q=quit")
inference_frame_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret or frame is None:
        print("Can't receive frame")
        break

    now = time.time()
    warmed_up = (now - app_start_time) >= WARMUP_SECONDS

    fps_n += 1
    if now - fps_t0 >= 1.0:
        fps = fps_n / (now - fps_t0)
        fps_t0 = now
        fps_n = 0

    h, w = frame.shape[:2]
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = detector.detect_for_video(mp_image, int(now * 1000))

    inference_frame_count += 1
    do_inference = (inference_frame_count % INFERENCE_SKIP_FRAMES == 0)

    is_closed_now = cached_eyes_closed
    is_yawn_now = cached_is_yawn

    head_pose_text = "NO FACE"
    distraction = False
    face_box = None

    if result.face_landmarks:
        lm = result.face_landmarks[0]
        try:
            head_pitch, head_yaw, head_roll = head_pose_from_landmarks(lm, w, h)
            yaw_history.append(head_yaw)
            pitch_history.append(head_pitch)
            head_yaw_s = sum(yaw_history) / len(yaw_history)
            head_pitch_s = sum(pitch_history) / len(pitch_history)

            if head_yaw_s < -YAW_THRESHOLD:
                head_pose_text = "LEFT"
            elif head_yaw_s > YAW_THRESHOLD:
                head_pose_text = "RIGHT"
            else:
                head_pose_text = "CENTER"

            if head_pitch_s > PITCH_THRESHOLD:
                head_pose_text += " / DOWN"
            elif head_pitch_s < -PITCH_THRESHOLD:
                head_pose_text += " / UP"

            distraction = abs(head_yaw_s) > YAW_THRESHOLD or abs(head_pitch_s) > PITCH_THRESHOLD
        except:
            distraction = False

        xs = [int(p.x * w) for p in lm]
        ys = [int(p.y * h) for p in lm]
        x1 = clamp(min(xs) - 20, 0, w - 1)
        y1 = clamp(min(ys) - 20, 0, h - 1)
        x2 = clamp(max(xs) + 20, 0, w)
        y2 = clamp(max(ys) + 20, 0, h)
        face_box = (x1, y1, x2 - x1, y2 - y1)
        last_face_box = face_box
        face_hold_frames = 0
    else:
        if last_face_box is not None and face_hold_frames < 8:
            face_box = last_face_box
            face_hold_frames += 1
        else:
            face_box = None
            last_face_box = None
            yaw_history.clear()
            pitch_history.clear()

    if face_box is not None:
        sx, sy, sfw, sfh = face_box
        face_crop = frame[sy:sy+sfh, sx:sx+sfw]
        if face_crop.size > 0:
            face_rgb_crop = cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB)
            face_pil = Image.fromarray(face_rgb_crop).resize((224, 224))
            face_tensor = eyes_transform(face_pil).unsqueeze(0).to(device)

            mouth_y1 = clamp(int(sy + sfh * 0.58), 0, h)
            mouth_y2 = clamp(int(sy + sfh * 0.98), 0, h)
            mouth_x1 = clamp(int(sx + sfw * 0.18), 0, w)
            mouth_x2 = clamp(int(sx + sfw * 0.82), 0, w)
            mouth_crop = frame[mouth_y1:mouth_y2, mouth_x1:mouth_x2]

            if do_inference and mouth_crop.size > 0:
                mouth_rgb = cv2.cvtColor(mouth_crop, cv2.COLOR_BGR2RGB)
                mouth_pil = Image.fromarray(mouth_rgb).resize((224, 224))
                mouth_tensor = yawn_transform(mouth_pil).unsqueeze(0).to(device)

                with torch.no_grad():
                    eyes_logits = eyes_model(face_tensor)
                    eyes_probs = torch.softmax(eyes_logits, 1)
                    eyes_pred = int(torch.argmax(eyes_probs, 1).item())
                    eyes_conf = float(eyes_probs.max().item())
                    raw_eye_closed = (eyes_pred == 0) and (eyes_conf > CONF_THRESHOLD)

                    if not distraction:
                        if stable_eye_state == raw_eye_closed:
                            stable_eye_frames += 1
                        else:
                            stable_eye_state = raw_eye_closed
                            stable_eye_frames = 1
                        if stable_eye_frames >= BLINK_STABLE_FRAMES:
                            prev = cached_eyes_closed
                            cached_eyes_closed = is_closed_now = raw_eye_closed
                            if prev and not is_closed_now:
                                blink_times.append(now)

                    yawn_logits = yawn_model(mouth_tensor)
                    yawn_probs = torch.softmax(yawn_logits, 1)
                    yawn_pred = int(torch.argmax(yawn_probs, 1).item())
                    yawn_conf = float(yawn_probs.max().item())
                    raw_yawn = (yawn_pred == 1) and (yawn_conf > YAWN_THRESHOLD)

                    if stable_yawn_state == raw_yawn:
                        stable_yawn_frames += 1
                    else:
                        stable_yawn_state = raw_yawn
                        stable_yawn_frames = 1
                    if stable_yawn_frames >= YAWN_STABLE_FRAMES:
                        prevy = cached_is_yawn
                        cached_is_yawn = is_yawn_now = raw_yawn
                        if prevy and not is_yawn_now:
                            yawn_times.append(now)

        x1, y1, fw1, fh1 = face_box
        cv2.rectangle(frame, (x1, y1), (x1 + fw1, y1 + fh1), (255, 200, 0), 2)

    if was_closed_prev and not is_closed_now and not distraction:
        blink_count += 1
    if was_yawn_prev and not is_yawn_now:
        yawn_count += 1

    if is_closed_now:
        if eyes_closed_since is None:
            eyes_closed_since = now
    else:
        eyes_closed_since = None

    was_closed_prev = is_closed_now
    was_yawn_prev = is_yawn_now

    closed_duration = 0.0 if eyes_closed_since is None else now - eyes_closed_since

    while blink_times and now - blink_times[0] > BLINK_WINDOW_SECONDS:
        blink_times.popleft()
    while yawn_times and now - yawn_times[0] > YAWN_WINDOW_SECONDS:
        yawn_times.popleft()

    blink_rate = len(blink_times)
    yawn_rate = len(yawn_times)

    fatigue_score = 0
    if warmed_up and blink_rate < BLINK_LOW_THRESHOLD:
        fatigue_score += 1
    if warmed_up and yawn_rate >= YAWN_HIGH_THRESHOLD:
        fatigue_score += 1
    if warmed_up and closed_duration >= CLOSED_ALARM_SECONDS:
        fatigue_score += 2

    fatigue_detected = fatigue_score >= 3

    overlay = frame.copy()
    cv2.rectangle(overlay, (15, 15), (740, 300), (20, 20, 20), -1)
    frame = cv2.addWeighted(overlay, 0.6, frame, 0.4, 0)

    if warmed_up and closed_duration >= CLOSED_ALARM_SECONDS:
        if not alarm_active:
            alarm_active = True
            alarm_start_time = now
            sleep_alarm_played = False

        flash = (0, 0, 255) if int(now * 5) % 2 else (255, 255, 255)
        cv2.rectangle(frame, (0, 0), (w, h), flash, 20)

        if not sleep_alarm_played:
            sleep_alarm_played = True
            start_alarm(ALARM_PATH, 1.0)

        if now - alarm_start_time > 2.0:
            alarm_active = False
            sleep_alarm_played = False
    else:
        alarm_active = False
        sleep_alarm_played = False

    if distraction:
        if distraction_since is None:
            distraction_since = now
    else:
        distraction_since = None

    if distraction_since is not None and (now - distraction_since) >= 1.0:
        distraction_msg_until = max(distraction_msg_until, now + 2.5)
        if not alarm_active:
            alarm_active = True
            alarm_start_time = now
            sleep_alarm_played = False
            start_alarm(ALARM_PATH, 1.0)

    pause_condition = warmed_up and fatigue_detected

    if pause_condition:
        if now >= pause_sound_cooldown_until:
            pause_sound_cooldown_until = now + 5.0
            if not pause_alert_played:
                pause_alert_played = True
                pause_alert_active = True
                pause_alert_until = now + 3.0
                start_alarm(PAUSE_ALARM_PATH, 1.0)
        pause_msg_until = max(pause_msg_until, now + 3.0)
    else:
        pause_alert_played = False
        pause_alert_active = False

    if now < pause_msg_until:
        label = "PAUZE AANGERADEN"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 1.1, 4)
        box_w = tw + 50
        box_h = th + 40
        mx1 = max((w - box_w) // 2, 0)
        my1 = max((h - box_h) // 2, 0)
        mx2 = min(mx1 + box_w, w)
        my2 = min(my1 + box_h, h)
        cv2.rectangle(frame, (mx1, my1), (mx2, my2), (0, 0, 0), -1)
        cv2.putText(frame, label, (mx1 + 25, my1 + th + 18),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.1, (0, 165, 255), 4, cv2.LINE_AA)

    if now < distraction_msg_until and not alarm_active:
        label = "LET OP DE WEG"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 1.0, 3)
        box_w = tw + 50
        box_h = th + 40
        dx1 = max((w - box_w) // 2, 0)
        dy1 = 60
        dx2 = min(dx1 + box_w, w)
        dy2 = min(dy1 + box_h, h)
        cv2.rectangle(frame, (dx1, dy1), (dx2, dy2), (0, 0, 0), -1)
        cv2.putText(frame, label, (dx1 + 25, dy1 + th + 18),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 3, cv2.LINE_AA)

    cv2.putText(frame, f'Blinks: {blink_count}', (25, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
    cv2.putText(frame, f'Yawns:  {yawn_count}', (25, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 200, 0), 2)
    cv2.putText(frame, f'Blink/min: {blink_rate}', (25, 110), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200, 200, 200), 2)
    cv2.putText(frame, f'Yawn/min:  {yawn_rate}', (25, 140), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200, 200, 200), 2)
    cv2.putText(frame, f'Eyes closed: {closed_duration:.1f}s', (25, 170), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200, 200, 200), 2)
    cv2.putText(frame, f'FPS: {fps:.1f}', (25, 200), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (150, 150, 150), 2)
    cv2.putText(frame, f'Head: {head_pose_text}', (25, 235), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 180, 80), 2)
    cv2.putText(frame, f'fatigue={fatigue_detected} score={fatigue_score}', (25, 265),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 255), 2)

    cv2.imshow("Drowsiness + Distraction Monitor", frame)

    key = cv2.waitKey(1) & 0xFF
    if key == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()
detector.close()
print("✅ Done")

Imports geladen!
🚀 MPS!
✅ Eyes OK
✅ Yawn OK


I0000 00:00:1778169901.526194 1997599 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M5
W0000 00:00:1778169901.532860 1997599 face_landmarker_graph.cc:174] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1778169901.543452 1999108 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778169901.552621 1999112 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


🎥 Drowsiness + Distraction Monitor | q=quit
✅ Done
